# 6b · Event-centric open extraction — n-ary nodes + relations  (per article)

The **event-centric twin** of the binary path. Instead of only relating pre-existing entities, this
notebook introduces **n-ary nodes** — reified nodes for the **events, observations, and claims** the
article is about — and then extracts **relations** (open, free-form, exactly as in `2_oie_extraction`
/ `6_relation_extraction`) that tie the participants to those n-ary nodes and to each other.

Everything a node connects to is a **relation** with an open `relation_phrase`; there are no fixed
"roles". Subjects and objects may be **entities** (`E..`, from nb4) or **n-ary nodes** (`EV../OBS../CL..`).
This file does **open extraction only** — it does not canonicalize. The free-form `relation_phrase`s
(and the n-ary node `type`s) are matched to schema.org / a **separate n-ary vocabulary** and minted
where nothing fits by the canonicalization stage (nb7, adapted), just like the binary path.

Reads the **same** entity nodes as nb6 (`typed_deduped_entities_ext_consolidated.json`, else
`typed_deduped_entities_ext.json`). Writes `results/<ARTICLE_ID>/nary_graph.json`. The binary path
(`6_relation_extraction` → `7`) is untouched, so the two methods stay comparable.

Set `ARTICLE_ID` for the single-article demo cell, or list ids in `SELECTED_ARTICLES` for the batch.

In [ ]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [89]:
# --- Config -------------------------------------------------------------------------------
from pathlib import Path

RESULTS_DIR  = ROOT / "results"            # reads results/<id>/typed_deduped_entities_ext*.json (nb4)
DATASET_DIR  = ROOT / "dataset/articles"   # reads <id>.txt for the REFERENCE_DATE and evidence spans
GEMINI_MODEL = "gemini-2.5-flash"

ARTICLE_ID   = "art5"                      # single article used by the demo cell below

# Which articles the BATCH cell processes, in one run.
#   None -> every article that has an entity file (typed_deduped_entities_ext*.json), sorted.
#   list -> exactly these ids, in this order.
SELECTED_ARTICLES = ["art4", "art5"]

# Which per-article entity file to use as participants (same as nb6). Consolidated preferred.
ENTITY_FILES = ["two_staged/1_entities/entities_typed_consolidated.json", "two_staged/1_entities/entities_typed.json"]

OUTPUT_NAME  = "two_staged/3_nary_graph/nary_graph.json"           # written to results/<id>/  (open, pre-canonicalization)

def entity_path(aid):
    for name in ENTITY_FILES:
        p = RESULTS_DIR / aid / name
        if p.exists():
            return p
    return None

_available = sorted({p.relative_to(RESULTS_DIR).parts[0] for p in RESULTS_DIR.glob("*/two_staged/1_entities/entities_typed*.json")})
if SELECTED_ARTICLES is None:
    ARTICLE_IDS = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    assert not missing, f"no entity file for: {missing}"
    ARTICLE_IDS = list(SELECTED_ARTICLES)

print(f"{len(_available)} article(s) have entity files; this run will process {ARTICLE_IDS}")

4 article(s) have entity files; this run will process ['art4', 'art5']


In [90]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time, re
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None

# Output token cap for one Gemini call.  None -> no limit (model maximum).  int -> cap the response.
MAX_OUTPUT_TOKENS = None
# gemini-2.5-flash "thinking" tokens SHARE the output budget; uncapped thinking on a big article can
# starve the JSON answer and truncate it.  0 -> thinking OFF (all budget to the answer; default).
THINKING_BUDGET = 0

def _loads_lenient(text, resp=None):
    """Parse Gemini's JSON, tolerating code fences / trailing commas, with a CLEAR message on truncation."""
    t = (text or "").strip()
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z]*\n?", "", t)
        t = re.sub(r"\n?```$", "", t).strip()
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        pass
    t2 = re.sub(r",(\s*[}\]])", r"\1", t)
    try:
        return json.loads(t2)
    except json.JSONDecodeError as e:
        fr = None
        try: fr = resp.candidates[0].finish_reason
        except Exception: pass
        raise ValueError(
            f"Gemini returned unparseable JSON (finish_reason={fr}; {len(t)} chars). "
            f"If finish_reason is MAX_TOKENS the output was TRUNCATED mid-JSON. Fixes: set THINKING_BUDGET=0 "
            f"(now {THINKING_BUDGET}) so thinking doesn't eat the budget, and/or raise MAX_OUTPUT_TOKENS "
            f"(now {MAX_OUTPUT_TOKENS}; None = model max). json error: {e}\n"
            f"...last 300 chars of output:\n{t[-300:]}"
        ) from None

def gemini_json(system_prompt, user_message, model=None, temperature=0, max_output_tokens="__default__"):
    """Call Gemini with JSON-mode output; return the parsed object and record latency."""
    global last_latency_s
    lim = MAX_OUTPUT_TOKENS if max_output_tokens == "__default__" else max_output_tokens
    cfg_kwargs = dict(system_instruction=system_prompt, temperature=temperature,
                      response_mime_type="application/json")
    if lim is not None:
        cfg_kwargs["max_output_tokens"] = lim
    if THINKING_BUDGET is not None and hasattr(genai_types, "ThinkingConfig"):
        cfg_kwargs["thinking_config"] = genai_types.ThinkingConfig(thinking_budget=THINKING_BUDGET)
    t0 = time.perf_counter()
    resp = client.models.generate_content(
        model=model or GEMINI_MODEL, contents=user_message,
        config=genai_types.GenerateContentConfig(**cfg_kwargs))
    last_latency_s = time.perf_counter() - t0
    return _loads_lenient(resp.text, resp)

Note: you may need to restart the kernel to use updated packages.


## 1. Load the article's entities (the participants relations can point at)

Identical to nb6, so the n-ary graph and the binary graph draw from the **same** entity node set. Each
entity is an id-stamped card (`E1, E2, …`). The n-ary nodes minted below (`EV.., OBS.., CL..`) are the
*additional* nodes that relations may also point at.

In [91]:
def _demojibake(s):
    """Repair UTF-8-decoded-as-latin-1 mojibake (e.g. 'Â£3.73m' -> '£3.73m'). Gated on tell-tale markers."""
    if isinstance(s, str) and any(m in s for m in ("Ã", "Â", "â€")):
        try:
            return s.encode("latin-1").decode("utf-8")
        except (UnicodeDecodeError, UnicodeEncodeError):
            return s
    return s

def load_entities(aid):
    """Read the article's final entity nodes (nb4) and build id-stamped participant cards."""
    p = entity_path(aid)
    assert p is not None, f"no entity file for {aid} (expected one of {ENTITY_FILES} under results/{aid}/)"
    data = json.loads(p.read_text())
    cards = []
    for i, e in enumerate(data.get("entities", []), 1):
        cards.append({"id": f"E{i}", "text": _demojibake(e.get("canonical_text")),
                      "type": e.get("chosen_type"), "aliases": [_demojibake(a) for a in e.get("aliases", [])],
                      "type_source": e.get("type_source"), "type_parent": e.get("type_parent"),
                      "type_vocab_id": e.get("vocab_id")})
    return p.name, cards

def _reference_date(text):
    """Parse the 'Date: DD/MM/YYYY' dateline -> ISO-8601 (YYYY-MM-DD), or None."""
    m = re.search(r"^Date:\s*(\d{2})/(\d{2})/(\d{4})", text, re.MULTILINE)
    if not m:
        return None
    d, mth, y = m.groups()
    return f"{y}-{mth}-{d}"

_fname, _cards = load_entities(ARTICLE_ID)
print(f"{ARTICLE_ID}: {len(_cards)} entities from {_fname}")
for c in _cards[:8]:
    print(f"  {c['id']:<4} {c['text']!r:<28} :: {c['type']}")

art5: 89 entities from typed_deduped_entities_ext_consolidated.json
  E1   'Ebola treatments trial'     :: MedicalTrial
  E2   'Democratic Republic of Congo' :: Country
  E3   'Belgium'                    :: Country
  E4   'UK'                         :: Country
  E5   'Uganda'                     :: Country
  E6   'France'                     :: Country
  E7   'Cape Verde'                 :: Country
  E8   'Argentina'                  :: Country


## 2. Open extraction — n-ary nodes + relations

One Gemini call per article. The model (1) mints **n-ary nodes** for the events / observations / claims
the article is about, and (2) extracts **open relations** — free-form `[subject, relation_phrase,
object]` triples where subject and object are entity or n-ary-node ids. No fixed role set and no
canonicalization here: the `relation_phrase`s and n-ary `type`s are matched to schema.org / a separate
n-ary vocabulary and minted downstream (nb7), exactly like the binary path.

In [92]:
NARY_SYSTEM_PROMPT = """You are an expert event-centric open information extraction system.
You are given a TEXT, its REFERENCE_DATE (publication date, ISO-8601), and a list of ENTITIES already extracted from it (each with an id, surface text, type, aliases).

GOAL: Build ONE concise, comprehensive, well-connected event-centric graph in two parts:
(1) N-ARY NODES — new reified nodes for the events, observations, and claims the TEXT is about; and
(2) RELATIONS — open, free-form directed relations tying nodes together, each with the relation phrase that best fits the TEXT. You are NOT given a fixed set of relations: invent the concise phrase that fits. After extraction, define each unique relation phrase.

N-ARY NODE KINDS (create nodes of these three kinds only):
- "Event": something that happens or is scheduled/forecast to happen.
- "Observation": a stated measurement, statistic, quantity, or quantified change.
- "Claim": a statement explicitly attributed to a source (someone says/states/reports/claims something).
Each n-ary node: {"id" (unique, e.g. EV1 / OBS1 / CL1), "node_kind", "type" (a concise free-form class label — PascalCase — as specific as the TEXT warrants), "label" (a SHORT natural title for the event/observation/claim — a few words, no packed-in facts), "is_main" (bool), "evidence" (verbatim span(s)), "confidence" (0-1)}.

RELATIONS: each is a directed triple {"subject_id","relation_phrase","object_id","evidence","confidence"} where subject_id and object_id are ids of ENTITIES or N-ARY NODES. Orient it so it reads subject --relation_phrase--> object. Choose the concise relation_phrase that best captures the link.

RULES:
1. Identify the article's MAIN event(s) and set "is_main": true on them.
2. Tie every participant of a stated fact to its n-ary node with a relation, and also capture the stable entity-to-entity relations. Aim for a single connected graph — do not leave a relevant node dangling.
3. Keep every fact's context connected to it: attach all the details that qualify a fact — its value, time, place, cause, qualifier, and source — to THAT fact's n-ary node, so the fact and its context form one connected cluster reachable in a few hops. Never leave a relation that adds context to a fact disconnected from that fact's node.
4. Keep each n-ary node's label a SHORT, natural title that reads as the event/observation/claim (a few words); do NOT pack facts into it - values, places, times, causes, quantities, sources and participants are RELATIONS, not label text. Any entity you name in the label MUST also be attached as a relation (the label never replaces an edge).
5. Give every Observation and Claim at least one relation to the Event it describes, and relate each Event to the other Events it is linked to in the TEXT if applicable (e.g. "part of" / "caused by" / "led to" / "followed"), so nothing hangs free of the event backbone.
6. When a breakdown gives several values that each belong to a distinct item (a category, group, period, or round), relate EACH value to its item AND link each item to the shared parent fact/event, so the whole breakdown stays in ONE connected cluster. Do NOT attach all the values and all the items as separate spokes on one summarizing node — that leaves you unable to tell which value goes with which item.
7. subject_id and object_id MUST be existing ENTITY or N-ARY NODE ids; never relate a node to itself.
8. Only omit entities that are not part of the article's actual content (navigation, "related"/"read more" links, boilerplate).
9. Extract only what the TEXT explicitly states. Use REFERENCE_DATE to resolve relative dates where possible.

OUTPUT strict JSON:
{
  "nary_nodes": [ {"id","node_kind","type","label","is_main","evidence":[...],"confidence"} ],
  "relations": [ {"subject_id","relation_phrase","object_id","evidence","confidence"} ],
  "relation_definitions": [ {"relation_phrase","definition"} ]
}
"""

def _fmt_entities(cards):
    return "\n".join(
        f'{c["id"]}\t{c["text"]}\t({c["type"]})' + (f'\taliases: {c["aliases"]}' if c["aliases"] else "")
        for c in cards)

def extract_nary(aid, write=True):
    """Open extraction of n-ary nodes + relations for one article -> results/<id>/nary_graph.json."""
    fname, cards = load_entities(aid)
    text = _demojibake((DATASET_DIR / f"{aid}.txt").read_text())
    ref_date = _reference_date(text)

    user_message = (
        f"REFERENCE_DATE: {ref_date}\n\n"
        f"ENTITIES (id\ttext\t(type)):\n{_fmt_entities(cards)}\n\n"
        f"TEXT:\n{text}"
    )
    out = gemini_json(NARY_SYSTEM_PROMPT, user_message)
    nary_nodes = out.get("nary_nodes", [])
    relations  = out.get("relations", [])

    # resolve each relation's endpoints to text/type (entity card OR n-ary node) so nb7 & the viz can
    # read subject/object surface + type, exactly like nb6's relation records.
    node_text = {c["id"]: (c["text"], c["type"]) for c in cards}
    for n in nary_nodes:
        node_text[n["id"]] = (n.get("label"), n.get("type"))
    for r in relations:
        r["subject_text"], r["subject_type"] = node_text.get(r.get("subject_id"), (None, None))
        r["object_text"], r["object_type"]   = node_text.get(r.get("object_id"), (None, None))

    graph = {
        "article_id":           aid,
        "reference_date":       ref_date,
        "entity_file":          fname,
        "entities":             cards,
        "nary_nodes":           nary_nodes,
        "relations":            relations,           # OPEN relation_phrases — canonicalized in nb7
        "relation_definitions": out.get("relation_definitions", []),
    }
    if write:
        p = RESULTS_DIR / aid / OUTPUT_NAME
        p.write_text(json.dumps(graph, indent=2, ensure_ascii=False))
        print(f"  wrote {p}  ({len(nary_nodes)} n-ary nodes, {len(relations)} relations)  [{last_latency_s:.1f}s]")
    return graph

## 3. Inspect the demo article

In [87]:
g = extract_nary(ARTICLE_ID)
by_id = {c["id"]: c["text"] for c in g["entities"]}
by_id.update({n["id"]: n["label"] for n in g["nary_nodes"]})

print("-" * 80)
print(f'{len(g["nary_nodes"])} n-ary nodes:')
for n in g["nary_nodes"]:
    star = " *MAIN*" if n.get("is_main") else ""
    print(f'  {n["id"]:<5} {n["node_kind"]:<12} <{n.get("type")}>{star}  "{n["label"]}"')
print("-" * 80)
print(f'{len(g["relations"])} relations (open relation_phrase — canonicalized later in nb7):')
for r in g["relations"]:
    s = by_id.get(r["subject_id"], r["subject_id"]); o = by_id.get(r["object_id"], r["object_id"])
    print(f'  {s!r}  --{r["relation_phrase"]}-->  {o!r}   (conf {r.get("confidence")})')
print("-" * 80)
print("relation_phrase definitions:")
for d in g["relation_definitions"]:
    print(f'  {d.get("relation_phrase"):<26} {d.get("definition","")}')

  wrote results/art5/nary_graph.json  (23 n-ary nodes, 97 relations)  [46.7s]
--------------------------------------------------------------------------------
23 n-ary nodes:
  EV1   Event        <Ebola Treatments Trial Commencement> *MAIN*  "Ebola treatments trial begins"
  CL1   Claim        <Announcement>  "WHO announced trial commencement"
  CL2   Claim        <Statement>  "Tedros on first patient enrollment"
  OBS1  Observation  <Ebola Case Statistics>  "Ebola cases and deaths in DR Congo"
  CL3   Claim        <Statement>  "Tedros on saving lives with therapeutics"
  CL4   Claim        <Statement>  "Dr Kamba on trial launch significance"
  EV2   Event        <Ebola Outbreak Commencement> *MAIN*  "Ebola outbreak began in May"
  OBS2  Observation  <Undetected Transmission>  "Ebola transmission undetected"
  EV3   Event        <Public Health Emergency Declaration>  "WHO declared public health emergency"
  OBS3  Observation  <Ebola Case Statistics (Updated)>  "Updated Ebola cases and 

## 4. Batch — process the selected articles (`ARTICLE_IDS`)

In [93]:
# Open extraction for each article. Re-runs overwrite. Canonicalization of relation_phrases + n-ary
# types into the separate n-ary vocabulary happens next, in nb7 (adapted for n-ary-node endpoints).
for aid in ARTICLE_IDS:
    print("=" * 60, aid)
    extract_nary(aid)
print(f"\ndone -- extracted {len(ARTICLE_IDS)} article(s) -> results/<id>/{OUTPUT_NAME}")

============================================================ art4
  wrote results/art4/nary_graph.json  (19 n-ary nodes, 67 relations)  [29.9s]
============================================================ art5
  wrote results/art5/nary_graph.json  (23 n-ary nodes, 97 relations)  [39.8s]

done -- extracted 2 article(s) -> results/<id>/nary_graph.json
